# Bronze — NHS cervical screening coverage

Lands the published source files unchanged, with provenance attached to every row.
No cleaning, no type conversion, no joins — those belong in Silver.

Run top to bottom. Requires the raw files fetched by `fetch_sources.sh`.

## 1. Paths and settings

In [30]:
from pathlib import Path
import hashlib, io, zipfile, urllib.request
from datetime import datetime, timezone

import pandas as pd

BASE_DIR     = Path.home() / "projects" / "nhs-cervical-screening-pipeline"
RAW_DIR      = BASE_DIR / "data" / "bronze" / "raw"
BRONZE_DIR   = BASE_DIR / "data" / "bronze"
EVIDENCE_DIR = BRONZE_DIR / "evidence"
TABLES_DIR   = BRONZE_DIR / "tables"

for d in (EVIDENCE_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# One timestamp for the whole run, so a run is identifiable as a single run.
RUN_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")

CAPTURE_EVIDENCE = True   # fetch source pages and the GOV.UK content API; needs internet
READ_GOVUK_ODS   = True   # the ODS files are large exported dashboards and read slowly

# Check the optional readers up front rather than failing halfway through the read loop.
missing = []
for module, package in [("pyarrow", "pyarrow"), ("odf", "odfpy")]:
    try:
        __import__(module)
    except ImportError:
        missing.append(package)

if missing:
    print(f"MISSING: {', '.join(missing)}   ->   pip install {' '.join(missing)}")
    if "odfpy" in missing:
        READ_GOVUK_ODS = False
        print("READ_GOVUK_ODS forced off: the GOV.UK regional tables will not be landed.")

print(BASE_DIR, "|", RUN_AT)

/Users/yusufismail/projects/nhs-cervical-screening-pipeline | 2026-09-02T23:27:10+00:00


## 2. Declare the sources

One entry per dataset I could read, including the four I decided against. Declared by hand rather
than read off the folder, so the rejections and the reasons are recorded.

`publication_date` is when the publisher released the file. `reference_period_end` is the date the
data describes. They are never the same column.

In [32]:
NHSD_ANNUAL_PAGE = ("https://digital.nhs.uk/data-and-information/publications/statistical/"
                    "cervical-screening-annual/england-2023-24")
NHSD_MI_PAGE     = ("https://digital.nhs.uk/data-and-information/publications/statistical/"
                    "cervical-screening-programme/"
                    "cervical-screening-programme-coverage-statistics-management-information")
GOVUK_2023_PAGE  = ("https://www.gov.uk/government/publications/"
                    "cervical-screening-quarterly-coverage-data-reports-2023")
GOVUK_2024_PAGE  = ("https://www.gov.uk/government/publications/"
                    "cervical-screening-quarterly-coverage-data-reports-2024")

ANNUAL_ZIP = "https://files.digital.nhs.uk/7E/4984C9/cervical-programme-annual-2023-24-csvs.zip"
MI_ZIP     = "https://files.digital.nhs.uk/A7/2F8C4D/cerv-scre-cov-stats-csv.zip"
ASSETS     = "https://assets.publishing.service.gov.uk/media"

# container : file under data/bronze/raw/ that was downloaded
# member    : file inside it when the container is a zip, otherwise None
# status    : used | rejected  (rejected rows are read by nobody, but stay as evidence)
SOURCES = [
    dict(source_id="annual_phof", series="annual", status="used", fmt="csv",
         publisher="NHS England Digital",
         publication_title="Cervical Screening Programme, England 2023-24",
         source_page_url=NHSD_ANNUAL_PAGE, file_url=ANNUAL_ZIP,
         container="annual_2023_24/cervical-programme-annual-2023-24-csvs.zip",
         member="phof.csv",
         publication_date="2024-11-28", publication_date_basis="stated on publication page",
         reference_period_end="2024-03-31",
         geography_levels="National; Region; Local Authority",
         note="Carries the 68.8% headline and the local authority spread."),

    dict(source_id="annual_kc53_coverage", series="annual", status="used", fmt="csv",
         publisher="NHS England Digital",
         publication_title="Cervical Screening Programme, England 2023-24",
         source_page_url=NHSD_ANNUAL_PAGE, file_url=ANNUAL_ZIP,
         container="annual_2023_24/cervical-programme-annual-2023-24-csvs.zip",
         member="kc53_coverage.csv",
         publication_date="2024-11-28", publication_date_basis="stated on publication page",
         reference_period_end="2024-03-31", geography_levels="National",
         note="England by fine age band, back to 2013-14. Includes bands outside the 25-64 "
              "invitation range, which must not enter a national coverage figure."),

    dict(source_id="annual_workbook", series="annual", status="rejected", fmt="xlsx",
         publisher="NHS England Digital",
         publication_title="Cervical Screening Programme, England 2023-24 - Data Tables",
         source_page_url=NHSD_ANNUAL_PAGE,
         file_url="https://files.digital.nhs.uk/7F/F4B9D8/"
                  "cervical-screening-programme-eng-2023-24-tab.xlsx",
         container="annual_2023_24/cervical-screening-programme-eng-2023-24-tab.xlsx",
         member=None,
         publication_date="2024-11-28", publication_date_basis="stated on publication page",
         reference_period_end="2024-03-31",
         geography_levels="National; Region; Local Authority; CCG",
         note="34 formatted presentation sheets. The CSVs in the same publication carry the same "
              "statistics already machine-readable. Rejected for convenience, not quality."),

    dict(source_id="mi_quarterly_la", series="mi_quarterly", status="used", fmt="csv",
         publisher="NHS England Digital",
         publication_title="Coverage Statistics [Management Information]",
         source_page_url=NHSD_MI_PAGE, file_url=MI_ZIP,
         container="mi_quarterly/cerv-scre-cov-stats-csv.zip",
         member="dashboard-phof-quarterly-la-2023-24-q4.csv",
         publication_date="2025-01-07", publication_date_basis="stated on publication page",
         reference_period_end="2024-03-31", geography_levels="Local Authority",
         note="13 quarters, 2020-21 Q4 to 2023-24 Q4. The LA series stops here: the publisher "
              "states LA data was unavailable from Q1 2024-25. Management information, not "
              "accredited official statistics."),

    dict(source_id="mi_quarterly_subicb", series="mi_quarterly", status="used", fmt="csv",
         publisher="NHS England Digital",
         publication_title="Coverage Statistics [Management Information]",
         source_page_url=NHSD_MI_PAGE, file_url=MI_ZIP,
         container="mi_quarterly/cerv-scre-cov-stats-csv.zip",
         member="dashboard-phof-quarterly-subICB-2024-25-q1.csv",
         publication_date="2025-01-07", publication_date_basis="stated on publication page",
         reference_period_end="2024-06-30", geography_levels="Sub-ICB",
         note="Runs one quarter beyond the LA file, to 2024-25 Q1 — the last public coverage "
              "observation of any kind. Extracted from NHAIS at closure on 24 June 2024 without "
              "the usual two-month lag, so it runs low and is not comparable with earlier quarters."),

    dict(source_id="mi_quarterly_gp", series="mi_quarterly", status="rejected", fmt="csv",
         publisher="NHS England Digital",
         publication_title="Coverage Statistics [Management Information]",
         source_page_url=NHSD_MI_PAGE, file_url=MI_ZIP,
         container="mi_quarterly/cerv-scre-cov-stats-csv.zip",
         member="dashboard-phof-quarterly-gp-2024-25-q1.csv",
         publication_date="2025-01-07", publication_date_basis="stated on publication page",
         reference_period_end="2024-06-30", geography_levels="GP practice",
         note="Below the smallest unit this project reasons about. Disclosure control applies at "
              "both GP practice and Sub-ICB level, but bites at practice grain, where counts are "
              "small across ~6,470 units; at Sub-ICB grain the sums reconcile to an independently "
              "published England total, so that file is used and this one is not."),

    dict(source_id="mi_q1_dq_issue", series="mi_quarterly", status="rejected", fmt="xlsx",
         publisher="NHS England Digital",
         publication_title="Q1 2024-25 Data Quality Issue - Duplicated GP practice codes",
         source_page_url=NHSD_MI_PAGE,
         file_url="https://files.digital.nhs.uk/E6/CBC257/"
                  "Q1%202024_25%20Data%20Quality%20Issue%20-%20Duplicated%20GP%20practices.xlsx",
         container="mi_quarterly/q1-2024-25-data-quality-issue.xlsx", member=None,
         publication_date="2025-02-13",
         publication_date_basis="stated in the correction notice on the publication page",
         reference_period_end="2024-06-30", geography_levels="GP practice",
         note="The 171 of 6,470 practices affected by a duplication error corrected 13 Feb 2025. "
              "Documents a correction to GP data, which is not used here."),

    dict(source_id="govuk_mar_2023", series="govuk_quarterly", status="used", fmt="ods",
         publisher="NHS England (published on GOV.UK)",
         publication_title="Cervical screening: quarterly coverage data reports 2023",
         source_page_url=GOVUK_2023_PAGE,
         file_url=f"{ASSETS}/653bd1c980884d0013f71ccb/"
                  "Cervical_Screening_Coverage_Report_Data_Tables_Mar_23.ods",
         container="govuk_quarterly/coverage_data_tables_Mar_23.ods", member=None,
         publication_date="2023-10-31",
         publication_date_basis="inferred: 2023 page first published; GOV.UK does not date "
                                "attachments individually",
         reference_period_end="2023-03-31", geography_levels="Region; Sub-ICB; GP practice",
         note="No local authority breakdown in any GOV.UK quarterly file, in any year."),

    dict(source_id="govuk_jun_2023", series="govuk_quarterly", status="used", fmt="ods",
         publisher="NHS England (published on GOV.UK)",
         publication_title="Cervical screening: quarterly coverage data reports 2023",
         source_page_url=GOVUK_2023_PAGE,
         file_url=f"{ASSETS}/653bc657d10f3500139a6ac6/"
                  "Cervical_Screening_Coverage_Report_Data_Tables_Jun_23.ods",
         container="govuk_quarterly/coverage_data_tables_Jun_23.ods", member=None,
         publication_date="2023-10-31",
         publication_date_basis="inferred: 2023 page first published",
         reference_period_end="2023-06-30", geography_levels="Region; Sub-ICB; GP practice",
         note=""),

    dict(source_id="govuk_sep_2023", series="govuk_quarterly", status="used", fmt="ods",
         publisher="NHS England (published on GOV.UK)",
         publication_title="Cervical screening: quarterly coverage data reports 2023",
         source_page_url=GOVUK_2023_PAGE,
         file_url=f"{ASSETS}/65c10043704282000d752245/"
                  "Cervical_Screening_Coverage_Report_Data_Tables_SEP_23.ods",
         container="govuk_quarterly/coverage_data_tables_Sep_23.ods", member=None,
         publication_date="2023-10-31",
         publication_date_basis="inferred: 2023 page first published",
         reference_period_end="2023-09-30", geography_levels="Region; Sub-ICB; GP practice",
         note="Published 31 days after the period it describes — the fastest in the series."),

    dict(source_id="govuk_dec_2023", series="govuk_quarterly", status="used", fmt="ods",
         publisher="NHS England (published on GOV.UK)",
         publication_title="Cervical screening: quarterly coverage data reports 2023",
         source_page_url=GOVUK_2023_PAGE,
         file_url=f"{ASSETS}/662a511055e1582b6ca7e611/"
                  "Cervical_Screening_Coverage_Report_Data_Tables_Dec_23.ods",
         container="govuk_quarterly/coverage_data_tables_Dec_23.ods", member=None,
         publication_date="2024-04-26",
         publication_date_basis="inferred: 2023 page last updated, the update adding December",
         reference_period_end="2023-12-31", geography_levels="Region; Sub-ICB; GP practice",
         note="Last quarter published on the 2023 page."),

    dict(source_id="govuk_mar_2024", series="govuk_quarterly", status="used", fmt="ods",
         publisher="NHS England (published on GOV.UK)",
         publication_title="Cervical screening: quarterly coverage data reports 2024",
         source_page_url=GOVUK_2024_PAGE,
         file_url=f"{ASSETS}/66d838847a73423428aa2f0f/"
                  "Cervical_Screening_Coverage_Report_Data_Tables_Mar_24.ods",
         container="govuk_quarterly/coverage_data_tables_Mar_24.ods", member=None,
         publication_date="2025-07-16",
         publication_date_basis="stated on the 2024 page; change history has one entry",
         reference_period_end="2024-03-31", geography_levels="Region; Sub-ICB; GP practice",
         note="The only file on the 2024 page, published 472 days after its period. The page is "
              "not listed in the GOV.UK collection index."),

    dict(source_id="govuk_mar_2024_subicb_gp", series="govuk_quarterly", status="rejected",
         fmt="xlsx", publisher="NHS England (published on GOV.UK)",
         publication_title="Cervical screening: quarterly coverage data reports 2024",
         source_page_url=GOVUK_2024_PAGE,
         file_url=f"{ASSETS}/66d8389bdbffedc7eb44d1ce/"
                  "Cervical_Screening_Coverage_Report_SubICB_GP_Mar2024_v1.0.xlsx",
         container="govuk_quarterly/coverage_subicb_gp_Mar_24.xlsx", member=None,
         publication_date="2025-07-16", publication_date_basis="stated on the 2024 page",
         reference_period_end="2024-03-31", geography_levels="Sub-ICB; GP practice",
         note="The Excel twin of the ODS on the same page. One is enough."),
]

manifest = pd.DataFrame(SOURCES)
manifest["run_at"] = RUN_AT

print(f"{len(manifest)} declared: {(manifest.status=='used').sum()} used, "
      f"{(manifest.status=='rejected').sum()} rejected")
manifest[["source_id", "series", "publication_date", "reference_period_end", "status"]]

13 declared: 9 used, 4 rejected


,source_id,series,publication_date,reference_period_end,status
0,annual_phof,annual,2024-11-28,2024-03-31,used
1,annual_kc53_coverage,annual,2024-11-28,2024-03-31,used
2,annual_workbook,annual,2024-11-28,2024-03-31,rejected
3,mi_quarterly_la,mi_quarterly,2025-01-07,2024-03-31,used
4,mi_quarterly_subicb,mi_quarterly,2025-01-07,2024-06-30,used
5,mi_quarterly_gp,mi_quarterly,2025-01-07,2024-06-30,rejected
6,mi_q1_dq_issue,mi_quarterly,2025-02-13,2024-06-30,rejected
7,govuk_mar_2023,govuk_quarterly,2023-10-31,2023-03-31,used
8,govuk_jun_2023,govuk_quarterly,2023-10-31,2023-06-30,used
9,govuk_sep_2023,govuk_quarterly,2023-10-31,2023-09-30,used


## 3. Capture the source pages

Some claims behind this project exist only as sentences on a web page, and pages get edited.
I keep a dated copy. GOV.UK also exposes a content API giving publication dates and attachment
URLs; NHS England Digital has no equivalent, so those are captured as HTML only.

In [34]:
EVIDENCE_TARGETS = {
    "page_nhsd_annual_2023_24.html": NHSD_ANNUAL_PAGE,
    "page_nhsd_mi_coverage.html":    NHSD_MI_PAGE,
    "page_nhsd_annual_series.html":  "https://digital.nhs.uk/data-and-information/publications/"
                                     "statistical/cervical-screening-annual",
    "page_govuk_collection.html":    "https://www.gov.uk/government/collections/"
                                     "cervical-screening-programme-data",
    "page_govuk_2023.html":          GOVUK_2023_PAGE,
    "page_govuk_2024.html":          GOVUK_2024_PAGE,
    "page_nhse_press_release.html":  "https://www.england.nhs.uk/2026/08/nhs-rolls-out-discreet-"
                                     "home-testing-kits-against-cervical-cancer-for-millions-of-women/",
    "api_govuk_collection.json":     "https://www.gov.uk/api/content/government/collections/"
                                     "cervical-screening-programme-data",
    "api_govuk_2023.json":           "https://www.gov.uk/api/content/government/publications/"
                                     "cervical-screening-quarterly-coverage-data-reports-2023",
    "api_govuk_2024.json":           "https://www.gov.uk/api/content/government/publications/"
                                     "cervical-screening-quarterly-coverage-data-reports-2024",
}

def capture(name, url, timeout=45):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "cervical-screening-pipeline/1.0"})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            body, status = r.read(), r.status
        (EVIDENCE_DIR / name).write_bytes(body)
        return dict(name=name, url=url, http_status=status, bytes=len(body),
                    sha256=hashlib.sha256(body).hexdigest(), captured_at=RUN_AT, error="")
    except Exception as e:
        return dict(name=name, url=url, http_status=None, bytes=0, sha256="",
                    captured_at=RUN_AT, error=f"{type(e).__name__}: {e}")

if CAPTURE_EVIDENCE:
    evidence = pd.DataFrame([capture(n, u) for n, u in EVIDENCE_TARGETS.items()])
    evidence.to_csv(EVIDENCE_DIR / "evidence_log.csv", index=False)
    failed = evidence[evidence.error != ""]
    print(f"captured {len(evidence) - len(failed)}/{len(evidence)}")
    if len(failed):
        print(failed[["name", "error"]].to_string(index=False))
else:
    evidence = pd.DataFrame()
    print("evidence capture skipped")

captured 10/10


## 4. Size and fingerprint each file

A changed hash on a later run means a publisher has amended something.

In [36]:
def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

# Loop over what was declared, not over the folder, so an undeclared file cannot slip through.
rows = []
for s in SOURCES:
    p = RAW_DIR / s["container"]
    rows.append(dict(source_id=s["source_id"],
                     file_path=str(p.relative_to(BASE_DIR)),
                     file_exists=p.exists(),
                     file_bytes=p.stat().st_size if p.exists() else 0,
                     sha256=sha256_of(p) if p.exists() else ""))

manifest = manifest.merge(pd.DataFrame(rows), on="source_id", how="left")
manifest[["source_id", "file_bytes", "sha256"]].assign(sha256=lambda d: d.sha256.str[:12])

,source_id,file_bytes,sha256
0,annual_phof,939732,41165d39c4bb
1,annual_kc53_coverage,939732,41165d39c4bb
2,annual_workbook,424211,9f0d1d662527
3,mi_quarterly_la,4775134,89d2d7f9a515
4,mi_quarterly_subicb,4775134,89d2d7f9a515
5,mi_quarterly_gp,4775134,89d2d7f9a515
6,mi_q1_dq_issue,17867,8d8279924551
7,govuk_mar_2023,1654005,969bff4faa19
8,govuk_jun_2023,1632850,7d9473ad5478
9,govuk_sep_2023,1552204,ec0cf0754259


## 5. Reconcile the folder against the manifest

Both directions. Unexpected files sort into three piles: operating system litter, archives unpacked
by hand while exploring, and anything else — which stops the run.

In [38]:
IGNORE_NAMES = {".DS_Store", "Thumbs.db", "desktop.ini"}

declared     = {(RAW_DIR / s["container"]).resolve() for s in SOURCES}
extract_dirs = {(RAW_DIR / s["container"]).parent / (RAW_DIR / s["container"]).stem
                for s in SOURCES if s["container"].endswith(".zip")}

ignored, extracted, unknown = [], [], []
for p in sorted(RAW_DIR.rglob("*")):
    if not p.is_file() or p.resolve() in declared:
        continue
    if p.name in IGNORE_NAMES or p.name.startswith("."):
        ignored.append(p)
    elif any(d in p.parents for d in extract_dirs):
        extracted.append(p)      # unpacked copy: present, unused, may drift from the zip
    else:
        unknown.append(p)

absent = sorted(d for d in declared if not d.exists())

print(f"declared {len(declared)} | ignored {len(ignored)} | "
      f"unpacked {len(extracted)} | unknown {len(unknown)}")
for p in unknown:
    print("  unknown:", p.relative_to(BASE_DIR))
for p in absent:
    print("  missing:", p.relative_to(BASE_DIR))

assert not absent,  "A declared source is missing. Re-run fetch_sources.sh."
assert not unknown, "Files in raw/ the manifest does not describe. Declare or remove them."

declared 10 | ignored 3 | unpacked 25 | unknown 0


## 6. Read and land

Everything reads as text. `keep_default_na=False` keeps blank, zero and suppressed values
distinguishable — deciding what they mean is Silver's job.

ODS files land as a raw grid with no header, because locating the start of the table is a
judgement call. One source in, one Parquet file out; no merging.

In [40]:
CSV_KWARGS = dict(dtype=str, keep_default_na=False, na_filter=False)

# Only the regional coverage sheets. The Sub-ICB and GP sheets are not needed here, and the
# Slicer_* sheets are artefacts of the tool that exported these dashboards.
ODS_SHEETS = ["Coverage_25to49_Region_Table", "Coverage_50to64_Region_Table"]

PROV = ["source_id", "series", "publisher", "publication_title", "source_page_url",
        "file_url", "publication_date", "publication_date_basis",
        "reference_period_end", "sha256"]

def read_csv_member(path, member):
    with zipfile.ZipFile(path) as z, z.open(member) as f:
        return pd.read_csv(io.TextIOWrapper(f, encoding="utf-8-sig"), **CSV_KWARGS)

def stamp(df, src, sha, member=""):
    """Every row carries its own provenance, so tracing a figure never needs a join."""
    for c in PROV:
        df["_" + c] = sha if c == "sha256" else src.get(c)
    df["_member"], df["_ingested_at"] = member, RUN_AT
    front = ["_" + c for c in PROV] + ["_member", "_ingested_at"]
    return df[front + [c for c in df.columns if c not in front]]

def save(df, name):
    out = TABLES_DIR / f"{name}.parquet"
    df.to_parquet(out, index=False)
    print(f"  {len(df):>7,} x {df.shape[1]:>3}  {out.name}")
    return dict(source_id=name.split("__")[0], rows=len(df),
                out_path=str(out.relative_to(BASE_DIR)))

# Each source is read inside try/except so one failure cannot leave `landed` half-built as a
# list. Failures are collected, then raised together at the end of the cell.
landed, read_errors = [], []

for src in [s for s in SOURCES if s["status"] == "used"]:
    sid, path = src["source_id"], RAW_DIR / src["container"]
    sha = manifest.loc[manifest.source_id == sid, "sha256"].iloc[0]
    print(sid)
    try:
        if src["fmt"] == "csv":
            df = (read_csv_member(path, src["member"]) if src["member"]
                  else pd.read_csv(path, **CSV_KWARGS))
            landed.append(save(stamp(df, src, sha, src["member"] or ""), sid))

        elif src["fmt"] == "ods" and READ_GOVUK_ODS:
            for sheet in ODS_SHEETS:
                df = pd.read_excel(path, sheet_name=sheet, engine="odf", header=None, dtype=str)
                landed.append(save(stamp(df, src, sha, sheet), f"{sid}__{sheet}"))
        else:
            print("  skipped")
    except Exception as e:
        read_errors.append(dict(source_id=sid, error=f"{type(e).__name__}: {e}"))
        print(f"  FAILED {type(e).__name__}: {e}")

landed = pd.DataFrame(landed, columns=["source_id", "rows", "out_path"])

if read_errors:
    print()
    print(pd.DataFrame(read_errors).to_string(index=False))
    raise RuntimeError(f"{len(read_errors)} source(s) failed to read — see above.")

annual_phof
      330 x  27  annual_phof.parquet
annual_kc53_coverage
      154 x  39  annual_kc53_coverage.parquet
mi_quarterly_la
    3,958 x  26  mi_quarterly_la.parquet
mi_quarterly_subicb
    2,756 x  29  mi_quarterly_subicb.parquet
govuk_mar_2023
       14 x  19  govuk_mar_2023__Coverage_25to49_Region_Table.parquet
       14 x  19  govuk_mar_2023__Coverage_50to64_Region_Table.parquet
govuk_jun_2023
       14 x  19  govuk_jun_2023__Coverage_25to49_Region_Table.parquet
       14 x  19  govuk_jun_2023__Coverage_50to64_Region_Table.parquet
govuk_sep_2023
       14 x  19  govuk_sep_2023__Coverage_25to49_Region_Table.parquet
       14 x  19  govuk_sep_2023__Coverage_50to64_Region_Table.parquet
govuk_dec_2023
       14 x  19  govuk_dec_2023__Coverage_25to49_Region_Table.parquet
       14 x  19  govuk_dec_2023__Coverage_50to64_Region_Table.parquet
govuk_mar_2024
       14 x  19  govuk_mar_2024__Coverage_25to49_Region_Table.parquet
       14 x  19  govuk_mar_2024__Coverage_50to64_Region_T

## 7. Checks

Values are converted to numbers inside the check and discarded — the stored tables stay text.

The mean of the local authority percentages is computed only to show what it costs. Coverage is a
ratio: sum the numerators, sum the denominators, divide once.

In [42]:
num = lambda s: pd.to_numeric(s, errors="coerce")

checks = []
def check(name, ok, detail=""):
    checks.append(dict(check=name, passed=bool(ok), detail=detail))

# Annual: the England row, the sum of the LA parts and the published figure must all agree.
phof = pd.read_parquet(TABLES_DIR / "annual_phof.parquet")
year = phof[phof.CollectionYearRange == "2023-24"]
la   = year[year.Org_Type == "LA"]

stated  = num(year[year.Org_Type == "National"].Coverage_PHOF_25_64).iloc[0]
n, d    = num(la.Screened_PHOF_42_66mth).sum(), num(la.Eligible_PHOF_25_64).sum()
derived = 100 * n / d
naive   = num(la.Coverage_PHOF_25_64).mean()

print(f"England row {stated:.4f}% | summed from {len(la)} LAs {derived:.4f}% | published 68.8%")
print(f"mean of LA percentages {naive:.4f}%  ->  wrong by {derived - naive:.2f}pp")

check("England row equals sum of LA parts", abs(stated - derived) < 1e-6)
check("reconciles to published 68.8%", round(derived, 1) == 68.8, f"{derived:.4f}")
check("151 local authorities", len(la) == 151, str(len(la)))
check("9 regions", (year.Org_Type == "Region").sum() == 9)

# Management information: both series stop where the manifest says they stop.
mi_la = pd.read_parquet(TABLES_DIR / "mi_quarterly_la.parquet")
mi_sb = pd.read_parquet(TABLES_DIR / "mi_quarterly_subicb.parquet")
la_last, sb_last = sorted(mi_la.Quarter.unique())[-1], sorted(mi_sb.Quarter.unique())[-1]
print(f"LA series ends {la_last} | Sub-ICB series ends {sb_last}")

check("LA series ends 2023-24 Q4", la_last == "2023-24 Q4", la_last)
check("Sub-ICB series ends 2024-25 Q1", sb_last == "2024-25 Q1", sb_last)

# Provenance is complete on everything landed.
for f in sorted(TABLES_DIR.glob("*.parquet")):
    t = pd.read_parquet(f)
    check(f"{f.stem}: provenance complete",
          t._publication_date.notna().all() and t._reference_period_end.notna().all())

results = pd.DataFrame(checks)
print(f"\n{results.passed.sum()}/{len(results)} checks passed")
if not results.passed.all():
    print(results[~results.passed].to_string(index=False))
assert results.passed.all(), "A Bronze check failed."

England row 68.7988% | summed from 151 LAs 68.7988% | published 68.8%
mean of LA percentages 68.5860%  ->  wrong by 0.21pp
LA series ends 2023-24 Q4 | Sub-ICB series ends 2024-25 Q1

20/20 checks passed


## 8. Write the manifest and summarise

Publication lag falls out of holding publication date and reference period as separate columns.

In [44]:
manifest_out = manifest.merge(
    landed.groupby("source_id").agg(tables=("out_path", "count"), rows=("rows", "sum")).reset_index(),
    on="source_id", how="left")
manifest_out[["tables", "rows"]] = manifest_out[["tables", "rows"]].fillna(0).astype(int)

manifest_out.to_parquet(BRONZE_DIR / "manifest.parquet", index=False)
manifest_out.to_csv(BRONZE_DIR / "manifest.csv", index=False)

print(f"run {RUN_AT} | {len(landed)} tables | {landed.rows.sum():,} rows | "
      f"{results.passed.sum()}/{len(results)} checks")
if len(evidence):
    print(f"evidence {(evidence.error == '').sum()}/{len(evidence)}")

lag = manifest_out[manifest_out.status == "used"][
    ["series", "source_id", "reference_period_end", "publication_date"]].copy()
lag["lag_days"] = (pd.to_datetime(lag.publication_date)
                   - pd.to_datetime(lag.reference_period_end)).dt.days
lag.sort_values("reference_period_end")

run 2026-09-02T23:27:10+00:00 | 14 tables | 7,338 rows | 20/20 checks
evidence 10/10


,series,source_id,reference_period_end,publication_date,lag_days
7,govuk_quarterly,govuk_mar_2023,2023-03-31,2023-10-31,214
8,govuk_quarterly,govuk_jun_2023,2023-06-30,2023-10-31,123
9,govuk_quarterly,govuk_sep_2023,2023-09-30,2023-10-31,31
10,govuk_quarterly,govuk_dec_2023,2023-12-31,2024-04-26,117
0,annual,annual_phof,2024-03-31,2024-11-28,242
1,annual,annual_kc53_coverage,2024-03-31,2024-11-28,242
3,mi_quarterly,mi_quarterly_la,2024-03-31,2025-01-07,282
11,govuk_quarterly,govuk_mar_2024,2024-03-31,2025-07-16,472
4,mi_quarterly,mi_quarterly_subicb,2024-06-30,2025-01-07,191
